In [60]:
import matplotlib.pyplot as plt
import pandas as pd
import sklearn

"""
Тип задачи: Бинарная классификация
Таргет y - Revenue совершил ли покупку пользователь
Признаки

Поведенческие метрики: Administrative, Informational, ProductRelated и соответствующие длительности (Duration).
Аналитические метрики страницы: BounceRates, ExitRates, PageValues.
Контекстные данные: SpecialDay, Month, OperatingSystems, Browser, Region, TrafficType, VisitorType, Weekend.
"""



df = pd.read_csv('./data/online_shoppers_intention.csv')


# Размер: (число строк, число колонок)
print(df.shape)

# Первые и случайные строки таблицы
df.head(5)
df.sample(5)

(12330, 18)


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
1332,0,0.0,0,0.0,1,0.000000,0.200000,0.200000,0.000000,0.0,Mar,1,1,1,3,Returning_Visitor,True,False
3247,0,0.0,0,0.0,23,255.638889,0.000000,0.005455,0.000000,0.0,May,3,2,4,2,Returning_Visitor,False,False
390,6,71.5,0,0.0,47,1532.200000,0.009929,0.019797,27.268880,0.0,Mar,2,2,1,1,Returning_Visitor,False,True
4932,2,57.0,0,0.0,28,724.066667,0.007407,0.017284,32.985333,0.0,May,1,1,1,4,Returning_Visitor,False,True
1523,1,27.5,0,0.0,11,243.666667,0.000000,0.018182,0.000000,0.0,Mar,3,2,1,3,Returning_Visitor,False,False


In [61]:
# EDA

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12330 entries, 0 to 12329
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Administrative           12330 non-null  int64  
 1   Administrative_Duration  12330 non-null  float64
 2   Informational            12330 non-null  int64  
 3   Informational_Duration   12330 non-null  float64
 4   ProductRelated           12330 non-null  int64  
 5   ProductRelated_Duration  12330 non-null  float64
 6   BounceRates              12330 non-null  float64
 7   ExitRates                12330 non-null  float64
 8   PageValues               12330 non-null  float64
 9   SpecialDay               12330 non-null  float64
 10  Month                    12330 non-null  str    
 11  OperatingSystems         12330 non-null  int64  
 12  Browser                  12330 non-null  int64  
 13  Region                   12330 non-null  int64  
 14  TrafficType              12330 no

In [62]:
df.duplicated().sum()
# удалить 125 дубликатов

np.int64(125)

In [63]:
df.isna().sum()
# nan нет

Administrative             0
Administrative_Duration    0
Informational              0
Informational_Duration     0
ProductRelated             0
ProductRelated_Duration    0
BounceRates                0
ExitRates                  0
PageValues                 0
SpecialDay                 0
Month                      0
OperatingSystems           0
Browser                    0
Region                     0
TrafficType                0
VisitorType                0
Weekend                    0
Revenue                    0
dtype: int64

In [64]:
df_only_zero = df[(df['Administrative'] == 0)
& (df['Informational'] == 0)
& (df['ProductRelated'] == 0)]

df_only_zero.info()
# удалить нулевые значения


<class 'pandas.DataFrame'>
Index: 6 entries, 2683 to 11865
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Administrative           6 non-null      int64  
 1   Administrative_Duration  6 non-null      float64
 2   Informational            6 non-null      int64  
 3   Informational_Duration   6 non-null      float64
 4   ProductRelated           6 non-null      int64  
 5   ProductRelated_Duration  6 non-null      float64
 6   BounceRates              6 non-null      float64
 7   ExitRates                6 non-null      float64
 8   PageValues               6 non-null      float64
 9   SpecialDay               6 non-null      float64
 10  Month                    6 non-null      str    
 11  OperatingSystems         6 non-null      int64  
 12  Browser                  6 non-null      int64  
 13  Region                   6 non-null      int64  
 14  TrafficType              6 non-null    

In [65]:
print(df[['ExitRates', 'BounceRates']].corr())
# высокая мультиколлеарность - признаки повторяют друг друга
# я удалю BounceRate так как ExitRate более полный признак


             ExitRates  BounceRates
ExitRates     1.000000     0.913004
BounceRates   0.913004     1.000000


In [66]:
# удалить нулевые значения
df = df.drop(df_only_zero.index)

# удаляю дубликаты
df = df.drop_duplicates()

# Просто убираем BounceRates из обучающего набора
df = df.drop(columns=['BounceRates'])
df = df.reset_index(drop=True)


df.info()




<class 'pandas.DataFrame'>
RangeIndex: 12199 entries, 0 to 12198
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Administrative           12199 non-null  int64  
 1   Administrative_Duration  12199 non-null  float64
 2   Informational            12199 non-null  int64  
 3   Informational_Duration   12199 non-null  float64
 4   ProductRelated           12199 non-null  int64  
 5   ProductRelated_Duration  12199 non-null  float64
 6   ExitRates                12199 non-null  float64
 7   PageValues               12199 non-null  float64
 8   SpecialDay               12199 non-null  float64
 9   Month                    12199 non-null  str    
 10  OperatingSystems         12199 non-null  int64  
 11  Browser                  12199 non-null  int64  
 12  Region                   12199 non-null  int64  
 13  TrafficType              12199 non-null  int64  
 14  VisitorType              12199 no

In [67]:
# count, mean, std, min, 25%, 50% (медиана), 75%, max
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Administrative,12199.0,2.340028,3.330851,0.0,0.000000,1.000000,4.000000,27.000000
Administrative_Duration,12199.0,81.686488,177.526253,0.0,0.000000,9.000000,94.750000,3398.750000
Informational,12199.0,0.508976,1.275880,0.0,0.000000,0.000000,0.000000,24.000000
Informational_Duration,12199.0,34.842583,141.457475,0.0,0.000000,0.000000,0.000000,2549.375000
ProductRelated,12199.0,32.061398,44.598950,0.0,8.000000,18.000000,38.000000,705.000000
ProductRelated_Duration,12199.0,1207.576104,1919.886750,0.0,193.708333,609.541667,1477.564759,63973.522230
ExitRates,12199.0,0.041389,0.046045,0.0,0.014223,0.025000,0.048466,0.200000
PageValues,12199.0,5.952500,18.657792,0.0,0.000000,0.000000,0.000000,361.763742
SpecialDay,12199.0,0.061972,0.199711,0.0,0.000000,0.000000,0.000000,1.000000
OperatingSystems,12199.0,2.124436,0.906839,1.0,2.000000,2.000000,3.000000,8.000000


In [68]:
print(df['Revenue'].value_counts(normalize=True))
# о 84% False и 16% True —  !PR-AUC!   /ROC-AUC вместо Accuracy

Revenue
False    0.843594
True     0.156406
Name: proportion, dtype: float64


In [69]:
cat_cols = ['Month', 'VisitorType', 'Weekend', 'OperatingSystems', 'Browser', 'Region', 'TrafficType']
for col in cat_cols:
    print(col, df[col].nunique())

Month 10
VisitorType 3
Weekend 2
OperatingSystems 8
Browser 13
Region 9
TrafficType 20


In [70]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler

# 1. Сокращение редких категорий в технических идентификаторах
def collapse_rare(series, top_n=3, other_label='Other'):
    top_categories = series.value_counts().nlargest(top_n).index
    return series.apply(lambda x: str(x) if x in top_categories else other_label)

# Преобразуем топ-категории (остальное схлопываем в 'Other')
df['OperatingSystems'] = collapse_rare(df['OperatingSystems'], top_n=3)
df['Browser'] = collapse_rare(df['Browser'], top_n=3)
df['TrafficType'] = collapse_rare(df['TrafficType'], top_n=5)
df['Region'] = collapse_rare(df['Region'], top_n=4)

# 2. Определение числовых и категориальных колонок
# BounceRates исключен ранее, PageValues убираем из-за Data Leakage
num_cols = [
    'Administrative', 'Administrative_Duration',
    'Informational', 'Informational_Duration',
    'ProductRelated', 'ProductRelated_Duration',
    'ExitRates', 'SpecialDay'
]

cat_cols = [
    'Month', 'OperatingSystems', 'Browser',
    'Region', 'TrafficType', 'VisitorType', 'Weekend'
]

df_clean = df.drop(columns=['PageValues'])
df_encoded = pd.get_dummies(df_clean, columns=cat_cols, drop_first=True)

# 4. Выделение признаков (X) и таргета (y)
X = df_encoded.drop(columns=['Revenue'])
y = df_encoded['Revenue'].astype(int)

# 5. Train / Test Split со стратификацией
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Размер обучающей выборки: {X_train.shape}")
print(f"Размер тестовой выборки:   {X_test.shape}")

# 6. Масштабирование числовых признаков (RobustScaler) для логистической регрессии
# Создаем копии датасетов для линейных моделей
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

scaler = RobustScaler()

# Обучаем скейлер ТОЛЬКО на train, трансформируем train и test
X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

Размер обучающей выборки: (9759, 35)
Размер тестовой выборки:   (2440, 35)


In [72]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    classification_report
)

# 1. Baseline (Dummy Classifier)
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)

# 2. Логистическая регрессия (с балансировкой классов и масштабированными признаками)
log_reg = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)

# 3. Дерево решений (без ограничений, покажем на нем переобучение)
dt_overfit = DecisionTreeClassifier(random_state=42)
dt_overfit.fit(X_train, y_train)

# Дерево с регуляризацией (ограничением глубины)
dt_tuned = DecisionTreeClassifier(max_depth=5, min_samples_leaf=20, random_state=42)
dt_tuned.fit(X_train, y_train)

# 4. Случайный лес (Ансамбль)
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)


# Функция для сбора метрик на Train и Test
def evaluate_model(name, model, x_tr, y_tr, x_te, y_te):
    # Предсказание вероятностей для ROC-AUC и PR-AUC
    if hasattr(model, "predict_proba"):
        prob_tr = model.predict_proba(x_tr)[:, 1]
        prob_te = model.predict_proba(x_te)[:, 1]
    else:
        prob_tr = [0] * len(y_tr)
        prob_te = [0] * len(y_te)

    pred_tr = model.predict(x_tr)
    pred_te = model.predict(x_te)

    return {
        'Model': name,
        'Train ROC-AUC': roc_auc_score(y_tr, prob_tr) if hasattr(model, "predict_proba") else 0.5,
        'Test ROC-AUC': roc_auc_score(y_te, prob_te) if hasattr(model, "predict_proba") else 0.5,
        'Train PR-AUC': average_precision_score(y_tr, prob_tr) if hasattr(model, "predict_proba") else y_tr.mean(),
        'Test PR-AUC': average_precision_score(y_te, prob_te) if hasattr(model, "predict_proba") else y_te.mean(),
        'Train F1': f1_score(y_tr, pred_tr, zero_division=0),
        'Test F1': f1_score(y_te, pred_te, zero_division=0)
    }


results = []
results.append(evaluate_model('Baseline (Dummy)', dummy, X_train, y_train, X_test, y_test))
results.append(evaluate_model('Logistic Regression', log_reg, X_train_scaled, y_train, X_test_scaled, y_test))
results.append(evaluate_model('Decision Tree (unconstrained)', dt_overfit, X_train, y_train, X_test, y_test))
results.append(evaluate_model('Decision Tree (max_depth=5)', dt_tuned, X_train, y_train, X_test, y_test))
results.append(evaluate_model('Random Forest', rf, X_train, y_train, X_test, y_test))

df_results = pd.DataFrame(results)
print(df_results.round(3).to_string(index=False))

                        Model  Train ROC-AUC  Test ROC-AUC  Train PR-AUC  Test PR-AUC  Train F1  Test F1
             Baseline (Dummy)          0.500         0.500         0.156        0.157     0.000    0.000
          Logistic Regression          0.755         0.751         0.331        0.336     0.396    0.396
Decision Tree (unconstrained)          1.000         0.574         1.000        0.189     1.000    0.284
  Decision Tree (max_depth=5)          0.771         0.746         0.370        0.334     0.216    0.225
                Random Forest          0.868         0.770         0.548        0.355     0.485    0.399


In [71]:
#

SyntaxError: invalid syntax (3496338796.py, line 4)